# MBRL Curvature — Colab GPU launcher

Runs the GPU side of the project (Mode A: full loop; Mode B: train on shards
collected locally). Designed for Colab Pro session death: checkpoints push to
W&B every `checkpoint.every` updates and `checkpoint.resume=auto` picks up the
newest one on relaunch — just re-run all cells.

**Runtime → Change runtime type → A100** (L4/T4 fine for Pendulum-class runs).

In [2]:
# 1. GPU sanity
import torch
print(torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

2.11.0+cpu | cuda: False | -


In [ ]:
# 2. Get the code. Git clone works in BOTH web Colab and the VS Code
# Colab extension; Drive mount needs the web frontend's auth widget.
REPO_URL = ""   # e.g. "https://github.com/you/mbrl-curvature.git"

if REPO_URL:
    !git clone -q $REPO_URL mbrl
    %cd mbrl
else:
    try:                       # web frontend only
        from google.colab import drive
        drive.mount('/content/drive')
        %cd /content/drive/MyDrive/mbrl
    except Exception as e:
        print("Drive mount unavailable (VS Code Colab extension?) ->", e)
        print("Set REPO_URL above, or upload the mbrl/ folder and %cd into it.")

!pip -q install -e ".[mujoco]" 2>&1 | tail -1

In [ ]:
# 3. W&B login — works on any frontend.
import os
key = os.environ.get("WANDB_API_KEY")
if not key:
    try:                       # web Colab Secrets (key icon), if available
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")
    except Exception:
        from getpass import getpass
        key = getpass("WANDB_API_KEY (from wandb.ai/authorize): ")
os.environ["WANDB_API_KEY"] = key
import wandb; wandb.login()

In [ ]:
# 4a. PHASE 0 — regression gate (doses from docs/original_findings_report.md):
# recipe on HalfCheetah, 3 seeds sharing this GPU. Pass criterion ~ +98 +- 23.
!python scripts/parallel_runs.py --preset colab_recipe \
    --overrides env=halfcheetah model.latent_dim=17 --seeds 0 1 2 --jobs 3

In [ ]:
# 4b. Mode B — import locally-collected replay shards first (optional)
# import wandb
# art = wandb.Api().artifact("you/mbrl-curvature/replay-HalfCheetah-v5:latest")
# shard_dir = art.download()
# then pass +buffer.shards=$shard_dir to train.py (wire-up in train.py when needed)

In [ ]:
# 5. Join a W&B sweep (GPU agent). Local CPU agents can join the same sweep id.
# SWEEP_ID = "you/mbrl-curvature/abc123"
# !wandb agent $SWEEP_ID

In [ ]:
# 6. Keep-alive / disconnect drill: simulate a kill and verify resume works.
# !timeout 120 python scripts/train.py seed=0   # dies after 2 min
# !python scripts/train.py seed=0 checkpoint.resume=auto   # must continue, not restart